# L5 Subset Recluster + LLM Sanity Check

This notebook supports targeted cleanup passes for potentially redundant `PROPOSED_LABEL` groups.

Workflow:
1. Load `PROPOSED_L5` + product attributes from `PRODUCTS_LCG`
2. Exclude noise rows where `DESCRIPTION` contains `INSERT`
3. Recluster each configured label-group subset
4. Generate cluster keyword summaries
5. (Optional) Ask Bedrock for proposed consolidated labels
6. (Optional) Run product-name sanity check and produce `LLM_LABEL`

In [1]:
from __future__ import annotations

import json
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    raise FileNotFoundError("Could not locate project root containing product_classifier_utils.py")

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import (  # noqa: E402
    build_product_text,
    embed_texts_from_cache,
    get_bedrock_client,
    get_snowflake_session,
    stable_text_hash,
)


In [9]:
DB = "SNOWFLAKE_LEARNING_DB"
SCHEMA = "SMCMAHON_PRODUCTS"
LABELS_TABLE = f"{DB}.{SCHEMA}.PROPOSED_L5"
PRODUCTS_TABLE = f"{DB}.{SCHEMA}.PRODUCTS_LCG"
JOIN_ID_COL = "PRODUCT_ID"
EXCLUDE_INSERT_PRODUCTS = True
ONLY_PRICED = False

TARGET_GROUPS = {
    "group_1_equipment_instruments": [
        "Lab Equipment",
        "Lab Instruments",
        "General Lab Equipment",
        "Lab Instruments and Tools",
        "Laboratory Hardware",
        "Analytical Balances",
        "Scientific Instruments",
    ],
    "group_2_furniture_storage": [
        "Laboratory Furniture",
        "Lab Seating",
        "Storage and Organization",
        "Lab Storage Solutions",
    ],
    "group_3_general_supplies": [
        "General Lab Supplies",
        "Lab Consumables",
    ],
    "group_4_filtration": [
        "Filtration Products",
        "Filter Papers",
        "Filtration Systems",
        "Filter Papers and Membranes",
    ],
}

AWS_PROFILE = "staging.admin"
AWS_REGION = "us-east-1"
EMBED_MODEL_ID = "amazon.titan-embed-text-v1"
MAX_WORKERS = 16
CHECKPOINT_EVERY = 2500
CACHE_PATH = PROJECT_ROOT / "artifacts/cache/embedding_cache.pkl"

PCA_COMPONENTS_TO_TRY = [None]
K_VALUES_TO_TRY = list(range(2, 11))
SAMPLE_SIZE_FOR_SILHOUETTE = 20000
RANDOM_STATE = 42

RUN_LLM_LABELING = True
# Use fallback list because Bedrock model availability varies by account/region.
LLM_MODEL_IDS = [
    "anthropic.claude-3-5-sonnet-20240620-v1:0",
    "anthropic.claude-3-haiku-20240307-v1:0",
]
LLM_TEMPERATURE = 0.2

RUN_LLM_SANITY_CHECK = True
SANITY_BATCH_SIZE = 40

SAVE_OUTPUT_FILES = False
OUT_DIR = PROJECT_ROOT / "artifacts" / "analysis" / "l5_subset_recluster"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
def _quote_sql(s: str) -> str:
    return str(s).replace("'", "''")


all_labels = sorted({x for vals in TARGET_GROUPS.values() for x in vals})
labels_sql = ", ".join([f"'{_quote_sql(lbl)}'" for lbl in all_labels])

where_parts = [f"l.PROPOSED_LABEL IN ({labels_sql})"]
if EXCLUDE_INSERT_PRODUCTS:
    where_parts.append("UPPER(COALESCE(p.DESCRIPTION, '')) NOT LIKE '%INSERT%'")
if ONLY_PRICED:
    where_parts.append("UPPER(COALESCE(p.PRICING_STATUS_C, '')) = 'PRICED'")
where_sql = " AND ".join(where_parts)

query = f"""
SELECT
  p.*,
  l.PROPOSED_LABEL,
  l.PROPOSED_CLUSTER
FROM {LABELS_TABLE} l
JOIN {PRODUCTS_TABLE} p
  ON p.{JOIN_ID_COL} = l.{JOIN_ID_COL}
WHERE {where_sql}
"""

sf = get_snowflake_session()
df = sf.sql(query).to_pandas()
df = df.drop_duplicates(subset=[JOIN_ID_COL]).reset_index(drop=True)

print(f"Rows loaded: {len(df):,}")
display(df["PROPOSED_LABEL"].value_counts().rename_axis("PROPOSED_LABEL").reset_index(name="ROW_COUNT"))


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://scienceexchange.okta.com/app/snowflake/exktzixqmxM7e7kdB697/sso/saml?SAMLRequest=lZJbb%2BIwEIX%2FSuR9TpxwKcECKlrEbirYIi6rZd9MMoCVxE49Dgn769eEsmofWqlv1vgc%2B5s5M7iv88w5gUah5JAEnk8ckLFKhDwMyWY9dUPioOEy4ZmSMCRnQHI%2FGiDPs4KNS3OUS3gpAY1jH5LImoshKbVkiqNAJnkOyEzMVuP5jLU8nxVaGRWrjLyxfO7giKCNJbxZEhQW72hMwSitqsqr2p7SB9ryfZ%2F6fWpVF8m3m762PX2gD6jfueitwsoXr2wPQl5H8BnW7ipC9mO9XriL59WaOOMb6qOSWOagV6BPIobNcnYFQEtQpHG31%2BmGXokucDRu4KFU1T7jKcQqL0pjn%2FXsie4hoZk6CNt5NBmSIhVJuE3y6PQ0P892z8FyHwZb0Rn%2Fbj2Vm%2B1MBdtTNDmH38%2B77rQfxsT5dYu2dYk2QiwhkpdAjS35rTvX77h%2Bbx2EzO%2BzTui128Ef4kxsoEJy0zhv1BgLOySAOj5yeQBPpYY3kLwo6H9%2BCnVq%2For6Ja%2FnPeilycNdv0cRFb3kTK6rwxoQPfryQAb0rf11DX%2FaZKLJQmUiPjtTpXNuPg4u8IKmIhJ330gZ5Fxk4yTRgGgDzDJVPWrgxm670SUQOrr%2B%2Bn7fR%2F8A&RelayState=ver%3A3-hint%3A9376132675904534-ETMsDgAAAZ1pIucqABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEPlzjVDONDSvH2oBbdlaS50AAACgNvma73EBGUyK%

,PROPOSED_LABEL,ROW_COUNT
0,Lab Equipment,19120
1,Lab Consumables,13774
2,General Lab Supplies,11778
3,General Lab Equipment,11303
4,Lab Instruments,11040
5,Laboratory Hardware,10790
6,Lab Storage Solutions,9963
7,Filtration Products,8293
8,Storage and Organization,7442
9,Lab Instruments and Tools,6787


In [10]:
def load_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("rb") as f:
        obj = pickle.load(f)
    if not isinstance(obj, dict):
        raise ValueError(f"Cache file {path} is not a dict")
    return obj


def save_cache(path: Path, cache: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("wb") as f:
        pickle.dump(cache, f)
    tmp_path.replace(path)


text_series = build_product_text(df)
texts = text_series.tolist()
text_hashes = [stable_text_hash(t) for t in texts]

cache = load_cache(CACHE_PATH)
cache_before = len(cache)
client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)


def checkpoint_cb(cache_obj: dict, processed_count: int) -> None:
    save_cache(CACHE_PATH, cache_obj)
    print(f"Checkpointed {processed_count:,} new embeddings. Cache size={len(cache_obj):,}")


X = embed_texts_from_cache(
    texts=texts,
    text_hashes=text_hashes,
    cache=cache,
    client=client,
    model_id=EMBED_MODEL_ID,
    show_progress=True,
    max_workers=MAX_WORKERS,
    checkpoint_every=CHECKPOINT_EVERY if CHECKPOINT_EVERY > 0 else None,
    on_checkpoint=checkpoint_cb if CHECKPOINT_EVERY > 0 else None,
)
save_cache(CACHE_PATH, cache)

print(f"Embedding matrix shape: {X.shape}")
print(f"Cache size: {cache_before:,} -> {len(cache):,}")
df["TEXT_TO_EMBED"] = text_series


Embedding matrix shape: (138864, 1536)
Cache size: 446,225 -> 446,225


In [11]:
def _top_terms_for_clusters(text_series: pd.Series, clusters: np.ndarray, top_n: int = 12) -> pd.DataFrame:
    vec = TfidfVectorizer(stop_words="english", max_features=50000)
    m = vec.fit_transform(text_series)
    terms = np.asarray(vec.get_feature_names_out())
    out = []
    for c in sorted(np.unique(clusters)):
        idx = np.where(clusters == c)[0]
        if len(idx) == 0:
            continue
        mean_scores = np.asarray(m[idx].mean(axis=0)).ravel()
        top_idx = np.argsort(-mean_scores)[:top_n]
        out.append({"cluster": int(c), "top_terms": terms[top_idx].tolist()})
    return pd.DataFrame(out)


def run_group_recluster(df_all: pd.DataFrame, X_all: np.ndarray, labels_for_group: list[str], group_name: str):
    mask = df_all["PROPOSED_LABEL"].isin(labels_for_group).to_numpy()
    df_g = df_all.loc[mask].reset_index(drop=True)
    X_g = X_all[mask]
    if len(df_g) < 200:
        raise ValueError(f"Group {group_name} too small: {len(df_g)} rows")

    records = []
    best = None
    rng = np.random.RandomState(RANDOM_STATE)

    for pca_n in PCA_COMPONENTS_TO_TRY:
        if pca_n is None:
            X_use = X_g
            pca_obj = None
            pca_name = "none"
        else:
            pca_obj = PCA(n_components=pca_n, random_state=RANDOM_STATE)
            X_use = pca_obj.fit_transform(X_g)
            pca_name = str(pca_n)

        for k in K_VALUES_TO_TRY:
            if k >= len(df_g):
                continue

            km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
            cl = km.fit_predict(X_use)
            inertia = float(km.inertia_)

            sil = np.nan
            if k > 1:
                sample_n = min(SAMPLE_SIZE_FOR_SILHOUETTE, len(df_g))
                if sample_n < len(df_g):
                    sample_idx = rng.choice(len(df_g), size=sample_n, replace=False)
                    sil = float(silhouette_score(X_use[sample_idx], cl[sample_idx]))
                else:
                    sil = float(silhouette_score(X_use, cl))

            rec = {
                "group": group_name,
                "pca": pca_name,
                "k": int(k),
                "silhouette": sil,
                "inertia": inertia,
            }
            records.append(rec)

            if not np.isnan(sil):
                if best is None or sil > best["silhouette"]:
                    best = {**rec, "clusters": cl, "df": df_g.copy(), "km": km, "pca_obj": pca_obj}

    if best is None:
        # Fallback in case only k=1 ran.
        first = records[0]
        km = KMeans(n_clusters=first["k"], random_state=RANDOM_STATE, n_init=20)
        cl = km.fit_predict(X_use)
        best = {**first, "clusters": cl, "df": df_g.copy(), "km": km, "pca_obj": pca_obj}

    best_df = best["df"].copy()
    best_df["SUB_CLUSTER"] = best["clusters"]
    top_terms_df = _top_terms_for_clusters(best_df["TEXT_TO_EMBED"], best_df["SUB_CLUSTER"].to_numpy(), top_n=12)

    metrics_df = pd.DataFrame(records).sort_values(["silhouette", "inertia"], ascending=[False, True], na_position="last")
    return metrics_df, best, best_df, top_terms_df


In [12]:
group_results = {}
summary_rows = []

for group_name, labels in TARGET_GROUPS.items():
    metrics_df, best, assigned_df, terms_df = run_group_recluster(df, X, labels, group_name)
    group_results[group_name] = {
        "labels": labels,
        "metrics": metrics_df,
        "best": best,
        "assigned": assigned_df,
        "terms": terms_df,
    }
    summary_rows.append({
        "group": group_name,
        "rows": int(len(assigned_df)),
        "best_pca": best["pca"],
        "best_k": int(best["k"]),
        "best_silhouette": float(best["silhouette"]),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("group")
display(summary_df)

for group_name in sorted(group_results.keys()):
    print(f"\n=== {group_name} ===")
    display(group_results[group_name]["metrics"].head(10))
    display(group_results[group_name]["terms"])


/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/stephanie.mcmahon/smcmahon_repo/.venv/lib/python3.10/site-packages/sklearn/clus

,group,rows,best_pca,best_k,best_silhouette
0,group_1_equipment_instruments,68709,none,6,0.092157
1,group_2_furniture_storage,23855,none,8,0.148906
2,group_3_general_supplies,25552,none,2,0.068908
3,group_4_filtration,20748,none,2,0.162876



=== group_1_equipment_instruments ===


,group,pca,k,silhouette,inertia
4,group_1_equipment_instruments,none,6,0.092157,4843629.5
7,group_1_equipment_instruments,none,9,0.092088,4669569.0
8,group_1_equipment_instruments,none,10,0.091616,4634711.0
5,group_1_equipment_instruments,none,7,0.090178,4749643.0
6,group_1_equipment_instruments,none,8,0.088824,4709075.0
1,group_1_equipment_instruments,none,3,0.088600,5204394.0
3,group_1_equipment_instruments,none,5,0.085786,4941540.0
2,group_1_equipment_instruments,none,4,0.085169,5052325.0
0,group_1_equipment_instruments,none,2,0.059953,5573533.0


,cluster,top_terms
0,0,"[size, manufacturer, price, pricing, status, l..."
1,1,"[size, ea, pc, product, quantity, priced, pric..."
2,2,"[high, cubis, tft, balance, touch, mca, color,..."
3,3,"[size, ea, product, quantity, pc, status, pric..."
4,4,"[size, pc, ea, product, quantity, status, pric..."
5,5,"[size, pc, ea, product, quantity, priced, pric..."


...

,group,pca,k,silhouette,inertia
0,group_3_general_supplies,none,2,0.068908,1905313.000
1,group_3_general_supplies,none,3,0.051386,1861722.375
2,group_3_general_supplies,none,4,0.050744,1831074.875
8,group_3_general_supplies,none,10,0.046368,1711875.375
7,group_3_general_supplies,none,9,0.044449,1725519.875
6,group_3_general_supplies,none,8,0.044352,1741307.375
5,group_3_general_supplies,none,7,0.041493,1760631.250
4,group_3_general_supplies,none,6,0.041490,1780183.250
3,group_3_general_supplies,none,5,0.037952,1803783.250


,cluster,top_terms
0,0,"[size, product, quantity, priced, list, descri..."
1,1,"[size, ea, product, quantity, priced, status, ..."



=== group_4_filtration ===


,group,pca,k,silhouette,inertia
0,group_4_filtration,none,2,0.162876,1347575.250
2,group_4_filtration,none,4,0.156034,1137305.750
3,group_4_filtration,none,5,0.147930,1090624.375
5,group_4_filtration,none,7,0.140998,1022217.750
6,group_4_filtration,none,8,0.138872,1001074.750
4,group_4_filtration,none,6,0.134278,1051403.375
1,group_4_filtration,none,3,0.134210,1227512.625
7,group_4_filtration,none,9,0.124877,981524.500
8,group_4_filtration,none,10,0.113065,960996.375


,cluster,top_terms
0,0,"[size, filter, product, description, status, p..."
1,1,"[size, ea, pc, quantity, priced, weight, tr, p..."


In [7]:
def _payload_text(payload: dict) -> str:
    parts = payload.get("content", [])
    chunks = []
    for p in parts:
        if isinstance(p, dict) and p.get("type") == "text":
            chunks.append(str(p.get("text", "")))
    if chunks:
        return "\n".join(chunks).strip()
    # Fallback for unexpected payload shapes
    return str(payload)


def _extract_json_obj(text: str) -> dict:
    import ast

    raw = str(text or "").strip()
    if not raw:
        raise ValueError("Empty LLM response text")

    # Remove markdown fences if present.
    raw = re.sub(r"^```(?:json)?", "", raw.strip())
    raw = re.sub(r"```$", "", raw.strip())

    # Try direct JSON parse first.
    try:
        return json.loads(raw)
    except Exception:
        pass

    # Try first JSON object slice.
    start = raw.find("{")
    end = raw.rfind("}")
    if start >= 0 and end > start:
        candidate = raw[start : end + 1]
        try:
            return json.loads(candidate)
        except Exception:
            try:
                obj = ast.literal_eval(candidate)
                if isinstance(obj, dict):
                    return obj
            except Exception:
                pass

    raise ValueError("Could not parse JSON object from LLM output")


def _discover_available_anthropic_models(region: str) -> list[str]:
    import boto3

    ctl = boto3.client("bedrock", region_name=region)
    resp = ctl.list_foundation_models(byProvider="Anthropic", byOutputModality="TEXT")
    out = []
    for m in resp.get("modelSummaries", []):
        mid = m.get("modelId")
        if not mid:
            continue
        # Prefer on-demand models because invoke_model uses on-demand by default.
        inf = set(m.get("inferenceTypesSupported", []))
        if inf and "ON_DEMAND" not in inf:
            continue
        out.append(mid)
    return sorted(set(out))


def invoke_llm_with_fallback(body: dict) -> tuple[dict, str]:
    last_err = None
    tried = []

    candidate_ids = list(LLM_MODEL_IDS)
    # Dynamic discovery helps when configured IDs are EOL/not-enabled.
    try:
        discovered = _discover_available_anthropic_models(AWS_REGION)
        preferred = [m for m in discovered if ("claude-3-5" in m or "claude-3" in m)]
        discovered_ordered = preferred + [m for m in discovered if m not in preferred]
        for m in discovered_ordered:
            if m not in candidate_ids:
                candidate_ids.append(m)
        if discovered_ordered:
            print(f"Discovered Anthropic models ({AWS_REGION}): {discovered_ordered[:6]}{' ...' if len(discovered_ordered) > 6 else ''}")
    except Exception as e:
        print(f"Model discovery warning (continuing with configured list): {e}")

    for model_id in candidate_ids:
        tried.append(model_id)
        try:
            resp = client.invoke_model(
                modelId=model_id,
                contentType="application/json",
                accept="application/json",
                body=json.dumps(body),
            )
            payload = json.loads(resp["body"].read().decode("utf-8"))
            return payload, model_id
        except Exception as e:
            last_err = e
            msg = str(e)
            if ("ResourceNotFound" in msg) or ("ValidationException" in msg) or ("AccessDenied" in msg) or ("Unsupported" in msg):
                continue
            raise

    raise RuntimeError(
        "No configured/discovered Anthropic model is available in this account/region. "
        f"Tried: {tried}. Last error: {last_err}"
    )


def suggest_labels_for_group(terms_df: pd.DataFrame, group_name: str) -> dict:
    prompt = {
        "task": "Propose concise consolidated labels for subclusters.",
        "group_name": group_name,
        "clusters": terms_df.to_dict(orient="records"),
        "format": {"cluster_to_label": {"0": "label"}, "label_descriptions": {"label": "short description"}},
    }
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1200,
        "temperature": LLM_TEMPERATURE,
        "messages": [{"role": "user", "content": [{"type": "text", "text": json.dumps(prompt)}]}],
    }
    payload, model_id_used = invoke_llm_with_fallback(body)
    raw = _payload_text(payload)
    try:
        parsed = _extract_json_obj(raw)
    except Exception:
        repair_prompt = {
            "task": "Repair the following model output into valid JSON only.",
            "raw_text": raw,
            "required_format": {
                "cluster_to_label": {"0": "label"},
                "label_descriptions": {"label": "short description"},
            },
        }
        repair_body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1200,
            "temperature": 0,
            "messages": [{"role": "user", "content": [{"type": "text", "text": json.dumps(repair_prompt)}]}],
        }
        repair_payload, _ = invoke_llm_with_fallback(repair_body)
        parsed = _extract_json_obj(_payload_text(repair_payload))
    parsed["_llm_model_used"] = model_id_used
    return parsed


llm_group_labels = {}
if RUN_LLM_LABELING:
    for group_name, obj in group_results.items():
        llm_group_labels[group_name] = suggest_labels_for_group(obj["terms"], group_name)
        print(f"Suggested labels ready for {group_name} using {llm_group_labels[group_name].get('_llm_model_used')}")

llm_group_labels


RuntimeError: No configured LLM model is available in this account/region. Tried: ['anthropic.claude-3-5-sonnet-20241022-v2:0']. Last error: An error occurred (ResourceNotFoundException) when calling the InvokeModel operation: This model version has reached the end of its life. Please refer to the AWS documentation for more details.

In [ ]:
CURRENT_CATEGORY_LIST_RAW = [
    "Hazardous Materials",
    "Shipping and Handling",
    "Safety and Compliance",
    "Chemical Storage",
    "Gloves and PPE",
    "Protective Wear",
    "First Aid Supplies",
    "Cleaning and Sterilization",
    "Labeling and Tape",
    "General Lab Supplies",
    "Lab Consumables",
    "Lab Accessories",
    "Pipettes and Tips",
    "Tubing and Hoses",
    "Microplates",
    "Media and Agar",
    "Microscopy Supplies",
    "Laboratory Bottles",
    "Caps and Closures",
    "Laboratory Glassware",
    "Tubes and Vials",
    "Pyrex Glassware",
    "Chemical Safety",
    "Chromatography Columns",
    "Syringes and Needles",
    "Medical Accessories",
    "Chemical Reagents",
    "Solutions and Solvents",
    "Chemicals and Reagents",
    "Organic Compounds",
    "Amino Acids",
    "Inorganic Compounds",
    "Biological Reagents",
    "Molecular Biology Kits",
    "Antibodies",
    "Filtration Products",
    "Filter Papers",
    "Filtration Systems",
    "Filter Papers and Membranes",
    "Laboratory Hardware",
    "Analytical Balances",
    "Scientific Instruments",
    "Lab Equipment",
    "Lab Instruments and Tools",
    "Laboratory Furniture",
    "Lab Seating",
    "Storage and Organization",
    "Lab Storage Solutions",
]


def normalize_label_text(s: str) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip())


def dedupe_labels(labels: list[str]) -> list[str]:
    seen = set()
    out = []
    for x in labels:
        n = normalize_label_text(x)
        if n and n not in seen:
            out.append(n)
            seen.add(n)
    return out


CURRENT_CATEGORY_LIST = dedupe_labels(CURRENT_CATEGORY_LIST_RAW)
print(f"Final category count: {len(CURRENT_CATEGORY_LIST)}")


def llm_sanity_check_labels(
    in_df: pd.DataFrame,
    candidate_labels: list[str],
    batch_size: int = 40,
) -> pd.DataFrame:
    """
    Strict sanity-check pass:
    - KEEP: label is correct
    - REASSIGN: choose different label from candidate list
    - REVIEW: uncertain / no good fit

    Returns new columns: LLM_DECISION, LLM_LABEL, LLM_REASON, LLM_CONFIDENCE, LLM_MATCH.
    """
    required_cols = {"PRODUCT_NAME", "PROPOSED_LABEL"}
    missing = sorted(required_cols - set(in_df.columns))
    if missing:
        raise ValueError(f"Missing required columns for sanity check: {missing}")

    allowed_labels = dedupe_labels(candidate_labels)
    allowed_set = set(allowed_labels)

    out = in_df.copy().reset_index(drop=True)
    out["PROPOSED_LABEL"] = out["PROPOSED_LABEL"].map(normalize_label_text)
    out["LLM_DECISION"] = None
    out["LLM_LABEL"] = None
    out["LLM_REASON"] = None
    out["LLM_CONFIDENCE"] = np.nan

    for start in range(0, len(out), batch_size):
        chunk = out.iloc[start : start + batch_size].copy()
        items = [
            {
                "row_id": int(i),
                "product_name": str(r["PRODUCT_NAME"]),
                "proposed_label": normalize_label_text(r["PROPOSED_LABEL"]),
            }
            for i, r in chunk.iterrows()
        ]

        prompt = {
            "task": (
                "Given product_name and proposed_label, decide whether to KEEP, REASSIGN, or REVIEW. "
                "REASSIGN must pick a label from candidate_labels. REVIEW if uncertain."
            ),
            "candidate_labels": allowed_labels,
            "rules": [
                "decision must be one of KEEP, REASSIGN, REVIEW",
                "llm_label must be one of candidate_labels or REVIEW",
                "if decision=KEEP, llm_label should equal proposed_label",
                "if decision=REVIEW, llm_label must be REVIEW",
                "confidence is float between 0 and 1",
            ],
            "items": items,
            "output_format": {
                "rows": [
                    {
                        "row_id": 1,
                        "decision": "KEEP|REASSIGN|REVIEW",
                        "llm_label": "label_or_REVIEW",
                        "reason": "short text",
                        "confidence": 0.0,
                    }
                ]
            },
        }

        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 3500,
            "temperature": 0,
            "messages": [{"role": "user", "content": [{"type": "text", "text": json.dumps(prompt)}]}],
        }

        payload, _model_id_used = invoke_llm_with_fallback(body)
        raw_text = _payload_text(payload)
        try:
            parsed = _extract_json_obj(raw_text)
        except Exception:
            repair_prompt = {
                "task": "Repair this response to valid JSON only.",
                "raw_text": raw_text,
                "required_format": {
                    "rows": [{"row_id": 1, "decision": "KEEP|REASSIGN|REVIEW", "llm_label": "label_or_REVIEW", "reason": "short text", "confidence": 0.0}]
                },
            }
            repair_body = {
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 3500,
                "temperature": 0,
                "messages": [{"role": "user", "content": [{"type": "text", "text": json.dumps(repair_prompt)}]}],
            }
            try:
                repair_payload, _ = invoke_llm_with_fallback(repair_body)
                parsed = _extract_json_obj(_payload_text(repair_payload))
            except Exception:
                parsed = {"rows": []}
                print(f"Warning: LLM JSON parse failed for batch starting at row {start}; defaulting unresolved rows to REVIEW.")

        rows = parsed.get("rows", []) if isinstance(parsed, dict) else []

        for row in rows:
            try:
                ridx = int(row.get("row_id"))
            except Exception:
                continue
            if ridx not in out.index:
                continue

            proposed = out.at[ridx, "PROPOSED_LABEL"]
            decision = str(row.get("decision", "REVIEW")).upper().strip()
            if decision not in {"KEEP", "REASSIGN", "REVIEW"}:
                decision = "REVIEW"

            llm_label = normalize_label_text(row.get("llm_label", "REVIEW"))
            if decision == "KEEP":
                llm_label = proposed if proposed in allowed_set else "REVIEW"
            elif decision == "REVIEW":
                llm_label = "REVIEW"
            elif llm_label not in allowed_set:
                decision = "REVIEW"
                llm_label = "REVIEW"

            reason = str(row.get("reason", "")).strip()[:240]
            try:
                conf = float(row.get("confidence", np.nan))
                if np.isfinite(conf):
                    conf = min(1.0, max(0.0, conf))
                else:
                    conf = np.nan
            except Exception:
                conf = np.nan

            out.at[ridx, "LLM_DECISION"] = decision
            out.at[ridx, "LLM_LABEL"] = llm_label
            out.at[ridx, "LLM_REASON"] = reason
            out.at[ridx, "LLM_CONFIDENCE"] = conf

    out["LLM_DECISION"] = out["LLM_DECISION"].fillna("REVIEW")
    out["LLM_LABEL"] = out["LLM_LABEL"].fillna("REVIEW")
    out["LLM_REASON"] = out["LLM_REASON"].fillna("")
    out["LLM_MATCH"] = out["LLM_LABEL"].eq(out["PROPOSED_LABEL"])
    return out


# Example quick-run (sample first):
# test_df = df[["PRODUCT_ID", "PRODUCT_NAME", "PROPOSED_LABEL"]].sample(200, random_state=RANDOM_STATE)
# test_out = llm_sanity_check_labels(test_df, CURRENT_CATEGORY_LIST, batch_size=SANITY_BATCH_SIZE)
# display(test_out.head(20))
# display(test_out["LLM_DECISION"].value_counts(dropna=False).rename_axis("decision").reset_index(name="count"))


Final category count: 48


In [ ]:
# Test on current list (safe sample first)
if RUN_LLM_SANITY_CHECK:
    SANITY_TEST_SAMPLE_SIZE = 300
    sanity_input = df[["PRODUCT_ID", "PRODUCT_NAME", "PROPOSED_LABEL"]].copy()
    sanity_test_df = sanity_input.sample(min(SANITY_TEST_SAMPLE_SIZE, len(sanity_input)), random_state=RANDOM_STATE)

    sanity_out = llm_sanity_check_labels(
        sanity_test_df,
        candidate_labels=CURRENT_CATEGORY_LIST,
        batch_size=SANITY_BATCH_SIZE,
    )

    print(f"Sanity test rows: {len(sanity_out):,}")
    display(
        sanity_out["LLM_DECISION"]
        .value_counts(dropna=False)
        .rename_axis("LLM_DECISION")
        .reset_index(name="ROW_COUNT")
    )
    display(
        sanity_out.groupby(["PROPOSED_LABEL", "LLM_LABEL"], dropna=False)
        .size()
        .rename("ROW_COUNT")
        .reset_index()
        .sort_values("ROW_COUNT", ascending=False)
        .head(40)
    )
    display(
        sanity_out[[
            "PRODUCT_ID",
            "PRODUCT_NAME",
            "PROPOSED_LABEL",
            "LLM_DECISION",
            "LLM_LABEL",
            "LLM_CONFIDENCE",
            "LLM_REASON",
        ]].head(30)
    )
else:
    print("Set RUN_LLM_SANITY_CHECK = True to run the test sample.")

Sanity test rows: 300


,LLM_DECISION,ROW_COUNT
0,KEEP,147
1,REVIEW,144
2,REASSIGN,9


,PROPOSED_LABEL,LLM_LABEL,ROW_COUNT
3,General Lab Equipment,REVIEW,31
13,Lab Instruments,REVIEW,27
8,Lab Equipment,Lab Equipment,25
6,Lab Consumables,Lab Consumables,22
5,General Lab Supplies,REVIEW,20
9,Lab Equipment,REVIEW,20
7,Lab Consumables,REVIEW,17
19,Lab Storage Solutions,Lab Storage Solutions,17
22,Laboratory Hardware,REVIEW,16
4,General Lab Supplies,General Lab Supplies,15


,PRODUCT_ID,PRODUCT_NAME,PROPOSED_LABEL,LLM_DECISION,LLM_LABEL,LLM_CONFIDENCE,LLM_REASON
0,01tPK00000Gcff8YAB,"Display Foil Quintix® Pro, 5 pcs.",Lab Consumables,REVIEW,REVIEW,NaN,
1,01tPP00000DJQ2DYAX,Vinyl Stool without Back - Medium Bench Height...,Lab Seating,REVIEW,REVIEW,NaN,
2,01tPP00000DJhuTYAT,"Thermco ASTM Certified Thermometers, Blue Spir...",Lab Instruments and Tools,REVIEW,REVIEW,NaN,
3,01tPP000005REAUYA4,"MYBATH 4L Digital Water Baths, Test tube rack ...",General Lab Supplies,REVIEW,REVIEW,NaN,
4,01tPK00000ICoSKYA1,SHELL/TANK PARTS ASY 5/6inch P.T,General Lab Equipment,REVIEW,REVIEW,NaN,
5,01tPK00000ICrpuYAD,SHELF BRACKET -D-,Storage and Organization,REVIEW,REVIEW,NaN,
6,01tPK00000GcZGKYA3,Set of 4 adapters 14 x 10/15 ml vac,General Lab Equipment,REVIEW,REVIEW,NaN,
7,01tPK00000ICx7PYAT,Calibration (4.1)elect. 12-CH,General Lab Supplies,REVIEW,REVIEW,NaN,
8,01tPK00000GcfPcYAJ,Microsart Validation Standard Achole,General Lab Supplies,REVIEW,REVIEW,NaN,
9,01tPP00000DJreZYAT,"Presence Absence Broth, 500 g",Lab Instruments,REVIEW,REVIEW,NaN,


In [ ]:
if SAVE_OUTPUT_FILES:
    summary_df.to_csv(OUT_DIR / "subset_group_best_configs.csv", index=False)
    for group_name, obj in group_results.items():
        obj["metrics"].to_csv(OUT_DIR / f"{group_name}_metrics.csv", index=False)
        obj["terms"].to_csv(OUT_DIR / f"{group_name}_cluster_terms.csv", index=False)
        obj["assigned"].to_csv(OUT_DIR / f"{group_name}_assigned_rows.csv", index=False)

    if RUN_LLM_LABELING:
        with (OUT_DIR / "group_llm_label_suggestions.json").open("w", encoding="utf-8") as f:
            json.dump(llm_group_labels, f, indent=2)

    print("Saved outputs to", OUT_DIR)
else:
    print("SAVE_OUTPUT_FILES=False: keeping results in notebook cell output only.")
